In [1]:
n = 318
number_of_topics = 18
n_clusters = 16
clustering_method = "birch"
trained_topic_model_folder = "Subphenotyping-for-PASC_test/trained_topic_model_cardiff_n318_v1"
print(f'results_cardiff_n{n}_{number_of_topics}topics_{n_clusters}clusters_v{trained_topic_model_folder.split("_v")[-1]}')

results_cardiff_n318_18topics_16clusters_v1


In [2]:
import numpy as np
import pandas as pd
import re
import os
import scipy.io as sio
import openpyxl
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)

import sklearn.cluster
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler
import umap

from matplotlib.lines import Line2D
from matplotlib.patches import Patch

import plotly.express as px
import plotly.graph_objects as go

# Loading files required to run this script

In [3]:
# Data dictionary mapping all hospitals symptoms together
# Changing the Cardiff naming to Sinai naming
file_ = "data_dictionary - Sheet1.csv"
data_dict = pd.read_csv(file_)
values = []
for col in data_dict["Cardiff Name"]: 
    col = str(col).replace(" ", "_").replace(".", "_").lower()
    while "__" in col: col = col.replace("__", "_")
    if col[-1] == "_": col = col[:-1]
    values.append(col)
data_dict["Cardiff Name"] = values
data_dict["Sinai Description"] = [str(val).lower() for val in data_dict["Sinai Description"]]
data_dict

,Organ System,Sinai Name,Sinai Description,Cardiff Name,Emory Name,UCSF Name,UCSF Description
0,Cognitive/Psych,currentsymptoms_27,"difficulty with concentration or reading or ""b...",poor_concentration,Brain Fog (3),concen,"trouble concentrating, trouble with your think..."
1,Cognitive/Psych,currentsymptoms_28,"confusion, difficulty thinking",nan,"Confusion, difficulty thinking",concen,"trouble concentrating, trouble with your think..."
2,Cognitive/Psych,currentsymptoms_30,memory problems or forgetfulness,nan,"Forgetful, memory problem",concen,"trouble concentrating, trouble with your think..."
3,Neurological,currentsymptoms_15,fatigue or tiredness,fatigue,Fatigue (1),fatig,feeling tired or having low energy
4,pulmonary,currentsymptoms_11,cough,cough,Cough,cough,cough
...,...,...,...,...,...,...,...
56,NaN,NaN,nan,nan,Thirst (3),NaN,NaN
57,NaN,NaN,nan,nan,Dry Eye,NaN,NaN
58,NaN,NaN,nan,nan,Rhinitis,NaN,NaN
59,Cognitive/Psych,NaN,nan,increased_anxiety_worry,Anxiety,NaN,NaN


In [4]:
# Getting trained topic model (topic_matrix, topic_proprtions)
file = f'{trained_topic_model_folder}/PFA_trained_model_n{n}_k{number_of_topics}.mat'
data = sio.loadmat(file) 
print(file)

# Getting sex, comorb, symptoms as `surveys`
surveys = pd.read_csv("data_cleaned/cardiff_data_all_n318.csv")
surveys = surveys.drop(298)
del surveys["id"]
del surveys["group"]

# for col in surveys.columns: 
#     if col not in mapping: 
#         print("Missing col in surveys", col)
#     else: 
#         surveys = surveys.rename(columns = {col: mapping[col]})
# # surveys = surveys.drop(labels = ["nan"], axis = 1)
surveys

Subphenotyping-for-PASC_test/trained_topic_model_cardiff_n318_v1/PFA_trained_model_n318_k18.mat


,chest_pain,palpitations_heart_racing,poor_concentration,breathlessness_shortness_of_breath,cough,fevers_did_you_feel_hot,headache,abdominal_pain,change_in_smell,dizziness_or_vertigo,muscle_pain,poor_balance_unsteadiness,sleep_disturbance,increased_anxiety_worry,low_mood_lack_of_enjoyment_in_normal_activities,nausea,vomiting,muscle_cramps,constipation,diarrhoea,feeling_faint_or_light_headed,loss_of_appetite,back_pain,chills_do_you_feel_unusually_cold,numbness_odd_or_loss_of_sensation,increased_body_odour_sweating,general_body_pain_e_g_arms_legs_or_joints,pain_during_sex_having_intercourse,rash,runny_or_congested_nose,sore_throat,high_temperature_when_you_need_it_measured,change_in_vision,fatigue
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0.0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0.0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0.0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0.0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0.0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
313,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0.0,0,0,0,0,0,0,0
314,1,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0.0,0,0,0,0,0,0,0
315,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0.0,0,0,0,0,0,0,0
316,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0.0,0,0,0,0,0,0,0


In [5]:
# surveys = pd.read_csv("cardiff_data_all_n318.csv")
# surveys[surveys["id"].isin(["0.093", "CO093"])]
# I think we go with 0.093 because 0 symptoms

In [6]:
# Getting color palette for umaps
colors = list(sns.color_palette("tab20"))
colors.extend(["purple", "black", "yellow", "green", "blue", "pink", "cyan", "cyan", "cyan"])

# symptoms = list(surveys.columns)
# file_ = open('symptoms_sinai.txt','r')
# symptoms = file_.readlines()[:44]
# file_.close()
# symptoms = [val.lower().replace("\n", "") for val in symptoms]
# print(len(symptoms), "symptoms")
symptoms = list(surveys.columns)
print("symptoms", len(symptoms), symptoms)

symptoms 34 ['chest_pain', 'palpitations_heart_racing', 'poor_concentration', 'breathlessness_shortness_of_breath', 'cough', 'fevers_did_you_feel_hot', 'headache', 'abdominal_pain', 'change_in_smell', 'dizziness_or_vertigo', 'muscle_pain', 'poor_balance_unsteadiness', 'sleep_disturbance', 'increased_anxiety_worry', 'low_mood_lack_of_enjoyment_in_normal_activities', 'nausea', 'vomiting', 'muscle_cramps', 'constipation', 'diarrhoea', 'feeling_faint_or_light_headed', 'loss_of_appetite', 'back_pain', 'chills_do_you_feel_unusually_cold', 'numbness_odd_or_loss_of_sensation', 'increased_body_odour_sweating', 'general_body_pain_e_g_arms_legs_or_joints', 'pain_during_sex_having_intercourse', 'rash', 'runny_or_congested_nose', 'sore_throat', 'high_temperature_when_you_need_it_measured', 'change_in_vision', 'fatigue']


In [7]:
# Getting symptom to organ system mapping
file_ = open('Neurological_Symptom_Mapping.txt','r')
mapping = [val.lower().replace("\n", "").replace("\t", "") for val in file_.readlines()]
file_.close()
neurologicals = [val.lower() for val in ["Neurological", "Cognitive/Psych", "Musculo-Skeletal", "G.I", "Sexual/Hormonal Function", "sensory changes", "temperature regulation", "pulmonary", "cardiovascular"]]
neurological_symptom_mapping, color_mapping, symptoms_ordered, symptoms_ordered_colors = {}, {}, [], []
curr, ct = "", -1
# for each line in Neurological_Symptom_Mapping
for val in mapping:
    val = val.replace("\t", "").lower()
#     print(val)
    # getting current symptom group
    if val in neurologicals: 
        curr = val
        ct += 1
    else: 
        if val == "abnormal changes in body temp": val = "Abnormal changes in body temperature".lower()
        elif val == "unexplained sweats or flushin": val = "Unexplained sweats or flushing".lower()
        elif val == "problems seeing (double or blurry vision)": 
            continue
        subset = data_dict[data_dict["Sinai Description"] == val]
        subset = subset[subset["Cardiff Name"] != "nan"]
        if subset.shape[0] == 2: 
            val_new = list(subset["Cardiff Name"])[0]
            if len(symptoms_ordered) > 0 and val_new == symptoms_ordered[-1]: 
                continue
            symptoms_ordered.append(val_new)
            symptoms_ordered_colors.append(colors[ct])
            color_mapping[curr] = colors[ct]
            val_new = list(subset["Cardiff Name"])[1]
            if len(symptoms_ordered) > 0 and val_new == symptoms_ordered[-1]: 
                continue
            symptoms_ordered.append(val_new)
            symptoms_ordered_colors.append(colors[ct])
            color_mapping[curr] = colors[ct]
        elif subset.shape[0] == 1: 
            val_new = list(subset["Cardiff Name"])[0]
            if len(symptoms_ordered) > 0 and val_new == symptoms_ordered[-1]: 
                continue
#             print(val, "::", val_new)
            symptoms_ordered.append(val_new)
            symptoms_ordered_colors.append(colors[ct])
            color_mapping[curr] = colors[ct]
#         else: print(val)
#         print(val_new, curr)
        if val_new == "low_mood_lack_of_enjoyment_in_normal_activities" and curr == "Cognitive/Psych".lower(): 
            val_new = "increased_anxiety_worry"
        elif val_new == "sleep_disturbance" and curr == "Neurological".lower(): 
            val_new = "poor_balance_unsteadiness"
        elif val_new == "loss_of_appetite" and curr == "G.I".lower(): 
            val_new = "constipation"
        elif val_new == "rash" and curr == "Musculo-Skeletal".lower(): 
            val_new = "back_pain"
        elif val_new == "increased_body_odour_sweating" and curr == "temperature regulation".lower(): 
            val_new = "runny_or_congested_nose"
        elif val_new == "numbness_odd_or_loss_of_sensation" and curr == "sensory changes".lower(): 
            val_new = "change_in_vision"
        if subset.shape[0] != 0 and val_new != list(subset["Cardiff Name"])[-1]: 
            symptoms_ordered.append(val_new)
            symptoms_ordered_colors.append(colors[ct])
            color_mapping[curr] = colors[ct]
# Filling in missing symptoms as other (black)
color_other = (0.75, 0.75, 0.75) # grey 
color_other = (0, 0, 0) # black
for symptom in symptoms: 
    if symptom not in symptoms_ordered: 
        symptoms_ordered.append(symptom)
        symptoms_ordered_colors.append(color_other)
        color_mapping["other"] = color_other
print(len(symptoms_ordered), "symptoms")
print(len(color_mapping.keys()),  "symptom groups")

FileNotFoundError: [Errno 2] No such file or directory: 'Neurological_Symptom_Mapping.txt'

In [ ]:
# Helper functions
def get_clusters(method, n_clusters, topic_proportions): 
    method = method.lower()
    if method == "kmeans": 
        clusters = sklearn.cluster.KMeans(n_clusters = n_clusters, random_state = 916).fit(topic_proportions)
        clusters = clusters.labels_ + 1
    elif method == "minibatch": 
        clusters = sklearn.cluster.MiniBatchKMeans(n_clusters = n_clusters, random_state = 916).fit(topic_proportions)
        clusters = clusters.labels_ + 1
    elif method == "spectralclustering": 
        clusters = sklearn.cluster.SpectralClustering(n_clusters = n_clusters, random_state = 916).fit(topic_proportions)
        clusters = clusters.labels_ + 1
    elif method == "birch": 
        clusters = sklearn.cluster.Birch(n_clusters = n_clusters).fit(topic_proportions)
        clusters = clusters.labels_ + 1
    elif method == "agglomerativeclustering": 
        clusters = sklearn.cluster.AgglomerativeClustering(n_clusters = n_clusters, random_state = 916).fit(topic_proportions)
        clusters = clusters.labels_ + 1
    elif method == "bisectingkmeans": 
        clusters = sklearn.cluster.BisectingKMeans(n_clusters = n_clusters, random_state = 916).fit(topic_proportions)
        clusters = clusters.labels_ + 1
    # Sorting by cluster sizes
    dictionary = sorted(dict(Counter(clusters)).items(), key=lambda x:x[1])
    dictionary.reverse()
    # [(1, 183), (3, 94), (5, 88), (9, 68), (2, 59), (8, 52), (7, 50), (4, 39), (6, 36)]
    mapping = dict(zip([val[0] for val in dictionary], np.unique(clusters))) # new:original
    # {1: 1, 3: 2, 5: 3, 9: 4, 2: 5, 8: 6, 7: 7, 4: 8, 6: 9}
    clusters = [mapping[val] for val in clusters]
    df = pd.DataFrame()
    df[0] = clusters
    dictionary = sorted(dict(Counter(clusters)).items(), key=lambda x:x[1])
    dictionary.reverse()
    return clusters

def plot_clusters(clusters, embedding, title, colors, topic_proportions_, highlights): 
    plt.figure()
    fig, ax = plt.subplots()
    plt.title(title)
    legend_elements = [] 
    for i in np.unique(clusters):
        a = plt.scatter(embedding[clusters==i, 0], embedding[clusters==i, 1], c = colors[i], s = 5)
        label = f"C{i}: {dict(Counter(clusters))[i]}"
        legend_elements.append(Line2D([0], [0], label = label, markerfacecolor = colors[i], markersize = 5, marker = 'o', color = 'w'))
        if highlights: 
            indeces = topic_proportions_[topic_proportions_["cluster"] == i].index
            indeces = list(set(indeces) & set(patients))
            a = plt.scatter(embedding[indeces, 0], embedding[indeces, 1], c = colors[i], s = 100, alpha = 0.5, marker = "^")
    ax.legend(handles = legend_elements, bbox_to_anchor=(1, 1))

# Topic Matrix Heatmap

In [ ]:
topic_matrix = pd.DataFrame(data['Phi_mean'])
print(topic_matrix.shape[0], "symptoms", topic_matrix.shape[1], "topics")
topic_matrix.columns = ["T" + str(val) for val in range(1, topic_matrix.shape[1] + 1)]
topic_matrix.index = pd.CategoricalIndex(symptoms, categories = symptoms_ordered)
topic_matrix = topic_matrix.sort_index()
# topic_matrix = topic_matrix.div(topic_matrix.sum(axis=1), axis=0) #### row-wise normalizing
print(topic_matrix.sum(0)[:3])
print(topic_matrix.sum(1)[:3])
ax = sns.clustermap(topic_matrix, vmin = 0, col_cluster = True, row_cluster = False, row_colors = symptoms_ordered_colors, 
                    standard_scale = 1, 
                    xticklabels = 1, yticklabels = 1, cbar_kws = {'label': ''}, cmap = "Greys", linecolor = "black", linewidth = 0.5, square = True)
handles = [Patch(facecolor=color_mapping[key]) for key in color_mapping.keys()]
plt.legend(handles, color_mapping, title='Symptom Groups',
           bbox_to_anchor=(0.95, 1.01), bbox_transform=plt.gcf().transFigure)
a = plt.setp(ax.ax_heatmap.get_xticklabels(), rotation=90, size = 10)
a = plt.title("col_scale") # 1
print(f'{file.split("Subphenotyping-for-PASC_test/")[-1]}')

In [ ]:
ax = sns.clustermap(topic_matrix, vmin = 0, col_cluster = True, row_cluster = False, row_colors = symptoms_ordered_colors, 
                    standard_scale = 0, 
                    xticklabels = 1, yticklabels = 1, cbar_kws = {'label': ''}, cmap = "Greys", linecolor = "black", linewidth = 0.5, square = True)
handles = [Patch(facecolor=color_mapping[key]) for key in color_mapping.keys()]
plt.legend(handles, color_mapping, title='Symptom Groups',
           bbox_to_anchor=(0.95, 1.01), bbox_transform=plt.gcf().transFigure)
a = plt.setp(ax.ax_heatmap.get_xticklabels(), rotation=90, size = 10)
a = plt.title("row_scale") # 0
print(f'{file.split("Subphenotyping-for-PASC_test/")[-1]}')

In [ ]:
ax = sns.clustermap(topic_matrix, vmin = 0, col_cluster = False, row_cluster = False, row_colors = symptoms_ordered_colors, 
                    standard_scale = 1, 
                    xticklabels = 1, yticklabels = 1, cbar_kws = {'label': ''}, cmap = "Greys", linecolor = "black", linewidth = 0.5, square = True)
handles = [Patch(facecolor=color_mapping[key]) for key in color_mapping.keys()]
plt.legend(handles, color_mapping, title='Symptom Groups',
           bbox_to_anchor=(0.95, 1.01), bbox_transform=plt.gcf().transFigure)
a = plt.setp(ax.ax_heatmap.get_xticklabels(), rotation=90, size = 10)
a = plt.title("col_scale") # 0
print(f'{file.split("Subphenotyping-for-PASC_test/")[-1]}')

In [ ]:
ax = sns.clustermap(topic_matrix, vmin = 0, col_cluster = False, row_cluster = False, row_colors = symptoms_ordered_colors, 
                    standard_scale = 0, 
                    xticklabels = 1, yticklabels = 1, cbar_kws = {'label': ''}, cmap = "Greys", linecolor = "black", linewidth = 0.5, square = True)
handles = [Patch(facecolor=color_mapping[key]) for key in color_mapping.keys()]
plt.legend(handles, color_mapping, title='Symptom Groups',
           bbox_to_anchor=(0.95, 1.01), bbox_transform=plt.gcf().transFigure)
a = plt.setp(ax.ax_heatmap.get_xticklabels(), rotation=90, size = 10)
a = plt.title("row_scale") # 0
print(f'{file.split("Subphenotyping-for-PASC_test/")[-1]}')

In [ ]:
ax = sns.clustermap(topic_matrix, vmin = 0, col_cluster = True, row_cluster = True, #row_colors = symptoms_ordered_colors, 
                    standard_scale = 1, 
                    xticklabels = 1, yticklabels = 1, cbar_kws = {'label': ''}, cmap = "Greys", linecolor = "black", linewidth = 0.5, square = True)
a = plt.setp(ax.ax_heatmap.get_xticklabels(), rotation=90, size = 10)
a = plt.title("col_scale") # 1
print(f'{file.split("Subphenotyping-for-PASC_test/")[-1]}')

In [ ]:
ax = sns.clustermap(topic_matrix, vmin = 0, col_cluster = True, row_cluster = True, #row_colors = symptoms_ordered_colors, 
                    standard_scale = 0, 
                    xticklabels = 1, yticklabels = 1, cbar_kws = {'label': ''}, cmap = "Greys", linecolor = "black", linewidth = 0.5, square = True)
a = plt.setp(ax.ax_heatmap.get_xticklabels(), rotation=90, size = 10)
a = plt.title("row_scale") # 0
print(f'{file.split("Subphenotyping-for-PASC_test/")[-1]}')

# Clustering

In [ ]:
topic_proportions = pd.DataFrame(data['Theta_mean']).T
topic_proportions = topic_proportions.drop(298) # manually removing duplicate patient
topic_proportions_norm = topic_proportions/topic_proportions.sum(1)[:,np.newaxis]
topic_proportions_norm = preprocessing.normalize(topic_proportions)
UMAP = umap.UMAP(metric='euclidean', n_neighbors=30, random_state=916)
embedding = UMAP.fit_transform(topic_proportions_norm)
clusters = get_clusters(clustering_method, n_clusters, topic_proportions)
# # for manually highlighting specific patients
# clusters[68] = 20
# clusters[298] = 20
topic_proportions_ = pd.DataFrame(topic_proportions).copy()
topic_proportions_["cluster"] = clusters
print(Counter(clusters))
print(f'{file.split("Subphenotyping-for-PASC_test/")[-1]}')
plot_clusters(clusters, embedding, f"{clustering_method} {n_clusters} clusters", colors, None, None)

In [ ]:
# Highlighting patients reporting 0 symptoms
title = "reported no symptoms"
patients = surveys.copy().reset_index(drop = True)
patients = list(patients[patients.sum(axis = 1) == 0.0].index)
plot_clusters(clusters, embedding, f"{len(patients)} out of {len(embedding)} patients {title}", colors, topic_proportions_, patients)

In [ ]:
# Highlighting control patients
title = "Case vs Control"
patients = pd.read_csv("data_cleaned/cardiff_data_all_n318.csv")
patients = patients.drop(298).reset_index(drop = True) # manually removing duplicate patient
patients = list(patients[patients["group"] == "Control"].index)
plot_clusters(clusters, embedding, f"{title}: {len(patients)} Control out of {len(embedding)} patients", colors, topic_proportions_, patients)

In [ ]:
# Highlighting case patients
title = "Case vs Control"
patients = pd.read_csv("data_cleaned/cardiff_data_all_n318.csv")
patients = patients.drop(298).reset_index(drop = True) # manually removing duplicate patient
patients = list(patients[patients["group"] == "Case"].index)
plot_clusters(clusters, embedding, f"{title}: {len(patients)} Case out of {len(embedding)} patients", colors, topic_proportions_, patients)

In [ ]:
# clusters_c1 = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 1, 1, 2, 1, 1, 2, 2, 1, 1, 1, 1, 2, 1, 2, 1, 2, 1, 1, 2, 2, 2, 2, 2, 2, 1, 2, 1, 1, 2, 2, 1, 1, 2, 2, 2, 2, 1, 1, 1, 2, 1, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
# clusters_c1 = [val - 1 for val in clusters_c1]
# print(clusters_c1)
# clusters_new = pd.DataFrame(clusters)
# values = []
# ct = 0
# for val in clusters_new[0]: 
#     if val == 1: 
#         val = clusters_c1[ct]
#         ct += 1
#     values.append(val)
# clusters_new[1] = values
# clusters_new[1].value_counts()

In [ ]:
# topic_proportions_["cluster"] = clusters_new[1]

In [ ]:
# def plot_clusters(clusters, embedding, title, colors, topic_proportions_, highlights): 
#     plt.figure()
#     fig, ax = plt.subplots()
#     plt.title(title)
#     legend_elements = [] 
#     for i in np.unique(clusters):
#         a = plt.scatter(embedding[clusters==i, 0], embedding[clusters==i, 1], c = colors[i], s = 5)
#         if i == 0: label = f"C1_a: {dict(Counter(clusters))[i]}"
#         elif i == 1: label = f"C1_b: {dict(Counter(clusters))[i]}"
#         else: label = f"C{i}: {dict(Counter(clusters))[i]}"
#         legend_elements.append(Line2D([0], [0], label = label, markerfacecolor = colors[i], markersize = 5, marker = 'o', color = 'w'))
#         if highlights: 
#             indeces = topic_proportions_[topic_proportions_["cluster"] == i].index
#             indeces = list(set(indeces) & set(patients))
#             a = plt.scatter(embedding[indeces, 0], embedding[indeces, 1], c = colors[i], s = 100, alpha = 0.5, marker = "^")
#     ax.legend(handles = legend_elements, bbox_to_anchor=(1, 1))
    
# # Highlighting specific patients
# title = "Case vs Control"
# patients = pd.read_csv("cardiff_data_all_n318.csv")
# patients = patients.drop(298)
# patients = list(patients[patients["group"] == "Control"].index)
# plot_clusters(clusters_new[1], embedding, f"{title}: {len(patients)} Control out of {len(embedding)} patients", colors, topic_proportions_, patients)

In [ ]:
# # Highlighting specific patients
# title = "Case vs Control"
# plot_clusters(clusters, embedding, f"{clustering_method} {n_clusters} clusters", list(severity["color"]), None, None)

In [ ]:
# Colored based on number of symptoms
plt.figure()
conditions = surveys[symptoms]
conditions["all"] = conditions.sum(axis = 1)
a = plt.scatter(embedding[:, 0], embedding[:, 1], c = conditions["all"], cmap = "binary", s = 5)
a = plt.title("number of symptoms")
a = plt.colorbar()

# Topic Proportions by Cluster

In [ ]:
topic_proportions = pd.DataFrame(data['Theta_mean'].T)
topic_proportions = topic_proportions.drop(298)
topic_proportions.columns = ["T" + str(val) for val in range(1, topic_proportions.shape[1] + 1)]
topic_proportions_ = topic_proportions.copy()
topic_proportions_["cluster"] = clusters
topic_proportions_bycluster = topic_proportions_.groupby("cluster").mean()
topic_proportions_bycluster.index = ["C" + str(val) for val in topic_proportions_bycluster.index]
topic_proportions_bycluster = topic_proportions_bycluster.T
print(topic_proportions_bycluster.shape[0], "topics",  topic_proportions_bycluster.shape[1], "clusters", topic_proportions.shape[0], "patients")
# topic_proportions_bycluster = topic_proportions_bycluster.div(topic_proportions_bycluster.sum(axis=1), axis=0) #### row-wise normalizing
# topic_proportions_bycluster = topic_proportions_bycluster.div(topic_proportions_bycluster.sum(axis=0), axis=1) #### column-wise normalizing
print(topic_proportions_bycluster.sum(0)[:3])
print(topic_proportions_bycluster.sum(1)[:3])

In [ ]:
ax = sns.clustermap(topic_proportions_bycluster, vmin = 0, col_cluster = False, row_cluster = True, figsize = (7, 7), 
#                     standard_scale = 1, 
                    xticklabels = 1, yticklabels = 1, cbar_kws = {'label': ''}, cmap = "Greys", linecolor = "black", linewidth = 0.5, square = True)
a = plt.title("no_scale") # None
a = plt.setp(ax.ax_heatmap.get_xticklabels(), rotation=0, size = 10)
print(f'{file.split("Subphenotyping-for-PASC_test/")[-1]}')

In [ ]:
ax = sns.clustermap(topic_proportions_bycluster, vmin = 0, col_cluster = False, row_cluster = True, figsize = (7, 7), 
                    standard_scale = 1, 
                    xticklabels = 1, yticklabels = 1, cbar_kws = {'label': ''}, cmap = "Greys", linecolor = "black", linewidth = 0.5, square = True)
a = plt.title("col_scale") # 1
a = plt.setp(ax.ax_heatmap.get_xticklabels(), rotation=0, size = 10)
print(f'{file.split("Subphenotyping-for-PASC_test/")[-1]}')

In [ ]:
ax = sns.clustermap(topic_proportions_bycluster, vmin = 0, col_cluster = False, row_cluster = True, figsize = (7, 7), 
                    standard_scale = 0, 
                    xticklabels = 1, yticklabels = 1, cbar_kws = {'label': ''}, cmap = "Greys", linecolor = "black", linewidth = 0.5, square = True)
a = plt.title("row_scale") # 0
a = plt.setp(ax.ax_heatmap.get_xticklabels(), rotation=0, size = 10)
print(f'{file.split("Subphenotyping-for-PASC_test/")[-1]}')

# Cluster by Symptom Heatmap

In [ ]:
cluster_by_symptom = pd.DataFrame(np.dot(topic_matrix, topic_proportions_bycluster))
print(cluster_by_symptom.shape, "=", topic_matrix.shape, "x", topic_proportions_bycluster.shape)
cluster_by_symptom.index = topic_matrix.index
cluster_by_symptom.columns = ["C" + str(val) for val in range(1, cluster_by_symptom.shape[1] + 1)]
# cluster_by_symptom = cluster_by_symptom.div(cluster_by_symptom.sum(axis=1), axis=0) #### row-wise normalizing
# cluster_by_symptom = cluster_by_symptom.div(cluster_by_symptom.sum(axis=0), axis=1) #### column-wise normalizing
print(cluster_by_symptom.sum(0)[:3])
print(cluster_by_symptom.sum(1)[:3])

In [ ]:
ax = sns.clustermap(cluster_by_symptom, vmin = 0, col_cluster = True, row_cluster = False, row_colors = symptoms_ordered_colors, 
#                     standard_scale = 1, 
                    xticklabels = 1, yticklabels = 1, cbar_kws = {'label': ''}, 
                    cmap = "Greys", linecolor = "black", linewidth = 0.5, square = True)
plt.legend(handles, color_mapping, title='Symptom Groups', facecolor = "white", 
           bbox_to_anchor=(0.95, 1.01), bbox_transform=plt.gcf().transFigure)
a = plt.setp(ax.ax_heatmap.get_xticklabels(), rotation=90, size = 12)
a = plt.title("no_scale") # 1
print(f'{file.split("Subphenotyping-for-PASC_test/")[-1]}')

In [ ]:
ax = sns.clustermap(cluster_by_symptom, vmin = 0, col_cluster = True, row_cluster = False, row_colors = symptoms_ordered_colors, 
                    standard_scale = 1, 
                    xticklabels = 1, yticklabels = 1, cbar_kws = {'label': ''}, 
                    cmap = "Greys", linecolor = "black", linewidth = 0.5, square = True)
plt.legend(handles, color_mapping, title='Symptom Groups', facecolor = "white", 
           bbox_to_anchor=(0.95, 1.01), bbox_transform=plt.gcf().transFigure)
a = plt.setp(ax.ax_heatmap.get_xticklabels(), rotation=90, size = 12)
a = plt.title("col_scale") # 1
print(f'{file.split("Subphenotyping-for-PASC_test/")[-1]}')

In [ ]:
ax = sns.clustermap(cluster_by_symptom, vmin = 0, col_cluster = True, row_cluster = False, row_colors = symptoms_ordered_colors, 
                    standard_scale = 0, 
                    xticklabels = 1, yticklabels = 1, cbar_kws = {'label': ''}, 
                    cmap = "Greys", linecolor = "black", linewidth = 0.5, square = True)
plt.legend(handles, color_mapping, title='Symptom Groups', facecolor = "white", 
           bbox_to_anchor=(0.95, 1.01), bbox_transform=plt.gcf().transFigure)
a = plt.setp(ax.ax_heatmap.get_xticklabels(), rotation=90, size = 12)
a = plt.title("row_scale") # 0
print(f'{file.split("Subphenotyping-for-PASC_test/")[-1]}')

In [ ]:
ax = sns.clustermap(cluster_by_symptom, vmin = 0, col_cluster = False, row_cluster = False, row_colors = symptoms_ordered_colors, 
#                     standard_scale = 1, 
                    xticklabels = 1, yticklabels = 1, cbar_kws = {'label': ''}, 
                    cmap = "Greys", linecolor = "black", linewidth = 0.5, square = True)
plt.legend(handles, color_mapping, title='Symptom Groups', facecolor = "white", 
           bbox_to_anchor=(0.89, 1.01), bbox_transform=plt.gcf().transFigure)
a = plt.setp(ax.ax_heatmap.get_xticklabels(), rotation=90, size = 12)
a = plt.title("no_scale") # 1
print(f'{file.split("Subphenotyping-for-PASC_test/")[-1]}')

In [ ]:
ax = sns.clustermap(cluster_by_symptom, vmin = 0, col_cluster = False, row_cluster = False, row_colors = symptoms_ordered_colors, 
                    standard_scale = 1, 
                    xticklabels = 1, yticklabels = 1, cbar_kws = {'label': ''}, 
                    cmap = "Greys", linecolor = "black", linewidth = 0.5, square = True)
plt.legend(handles, color_mapping, title='Symptom Groups', facecolor = "white", 
           bbox_to_anchor=(0.89, 1.01), bbox_transform=plt.gcf().transFigure)
a = plt.setp(ax.ax_heatmap.get_xticklabels(), rotation=90, size = 12)
a = plt.title("col_scale") # 1
print(f'{file.split("Subphenotyping-for-PASC_test/")[-1]}')

In [ ]:
ax = sns.clustermap(cluster_by_symptom, vmin = 0, col_cluster = False, row_cluster = False, row_colors = symptoms_ordered_colors, 
                    standard_scale = 0, 
                    xticklabels = 1, yticklabels = 1, cbar_kws = {'label': ''}, 
                    cmap = "Greys", linecolor = "black", linewidth = 0.5, square = True)
plt.legend(handles, color_mapping, title='Symptom Groups', facecolor = "white", 
           bbox_to_anchor=(0.89, 1.01), bbox_transform=plt.gcf().transFigure)
a = plt.setp(ax.ax_heatmap.get_xticklabels(), rotation=90, size = 12)
a = plt.title("row_scale") # 0
print(f'{file.split("Subphenotyping-for-PASC_test/")[-1]}')

# Defining symptom severity groups

In [ ]:
# Standardize the data
# https://github.com/mwaskom/seaborn/blob/86b5481ca47cb46d3b3e079a5ed9b9fb46e315ef/seaborn/matrix.py#L892
standardized = cluster_by_symptom.copy().T
subtract = standardized.min()
standardized = (standardized - subtract) / (standardized.max() - standardized.min())
standardized = standardized.T
standardized
# # Plotting to check same as sns.clustermap standard_scaling
# ax = sns.clustermap(standardized, vmin = 0, col_cluster = True, row_cluster = False, row_colors = symptoms_ordered_colors, 
# #                     standard_scale = 0, 
#                     xticklabels = 1, yticklabels = 1, cbar_kws = {'label': ''}, 
#                     cmap = "Greys", linecolor = "black", linewidth = 0.5, square = True)

In [ ]:
severity = pd.DataFrame(standardized.sum()).rename(columns = {0: "sum"}).sort_values("sum").reset_index()
a = plt.scatter(range(1, severity.shape[0] + 1), severity["sum"])

In [ ]:
severity = pd.DataFrame(standardized.sum()).rename(columns = {0: "sum"}).sort_values("sum").reset_index()
severity["cluster"] = [int(val.replace("C", "")) for val in severity["index"]]
# Manually defining symptom groups based on plot above
colors_ = ["green"]*1
colors_.extend(["yellow"]*12)
colors_.extend(["red"]*3)
severity["color"] = colors_
for index, row in severity.iterrows(): 
    a = plt.scatter([index], row["sum"], c = row["color"])
a = plt.ylabel("Severity Score")
a = plt.xticks(range(0, severity.shape[0]), severity["index"])

In [ ]:
# severity.to_csv(f"severity_cardiff_n{n}_{n_clusters}clusters.csv", index = False)
severity

# Star Charts per Cluster

In [ ]:
patient_by_symptom = pd.DataFrame(np.dot(topic_matrix, topic_proportions.T))
print(patient_by_symptom.shape,  "=", topic_matrix.shape, "+", topic_proportions.T.shape)
patient_by_symptom.index = topic_matrix.index
patient_by_symptom = patient_by_symptom.T
# patient_by_symptom = patient_by_symptom.div(patient_by_symptom.sum(axis=1), axis=0) #### row-wise normalizing
# patient_by_symptom = patient_by_symptom.div(patient_by_symptom.sum(axis=0), axis=1) #### column-wise normalizing
print(patient_by_symptom.sum(0)[:3])
print(patient_by_symptom.sum(1)[:3])
patient_by_symptom["cluster"] = clusters
patient_by_symptom
df = pd.DataFrame()
cluster_by_symptom_norm = cluster_by_symptom.div(cluster_by_symptom.sum(axis=1), axis=0) #### row-wise normalizing
for col in cluster_by_symptom_norm.columns: 
    subset = cluster_by_symptom_norm[col].sort_values(ascending = False)
    df[col + "_sym"] = subset.index
    df[col + "_pro"] = [round(val, 3) for val in list(subset)]
columns = list(filter(re.compile(".*sym").match, df.columns))
cluster_topsymptoms = df[columns].copy()
cluster_topsymptoms.columns = [col.split("_")[0] for col in cluster_topsymptoms.columns]
cluster_topsymptoms = cluster_topsymptoms.iloc[:6, :]
cluster_topsymptoms

In [ ]:
dictionary = {}
for index, row in patient_by_symptom.iterrows(): 
    cluster = "C" + str(int(row["cluster"]))
    symptoms_subset = list(cluster_topsymptoms[cluster])[:6]
    symptoms_subset.append(symptoms_subset[0])
    values = [row[s] for s in symptoms_subset]
    values.append(values[0])
    dictionary[index] = dict(r = values, theta = symptoms_subset)
print(len(dictionary))
# dictionary

In [ ]:
# Adding sexes if provided
surveys = pd.read_csv("data_cleaned/cardiff_data_all_n318.csv")
surveys["id"] = [val.replace("CO093", "C0093").replace("0.", "CO").replace("1.", "CA") for val in surveys["id"]]
values = []
for val in surveys["id"]: 
    while len(val) < 5: 
        val = val + "0"
    values.append(val)
surveys["id"] = values
surveys = surveys[surveys["id"] != "C0093"].reset_index(drop = True)
metadata = pd.read_csv("data_cleaned/metadata_cardiff_n338.csv")
surveys = surveys.merge(metadata[["id", "sex"]], on = "id", how = "left")
# print(dict(surveys["sex"].value_counts()))
# count_sex = {"rgba(0,255,0,0.25)": 0, "rgba(0,0,255,0.25)": 0, "rgba(50,50,50,0.25)": 0}
# count_sex = {"rgb(0,255,0)": 0, "rgb(0,0,255)": 0, "rgb(50,50,50)": 0}
count_sex = {"rgb(0,0,255)": 0, "rgb(255,0,0)": 0, "rgb(50,50,50)": 0}
# values = ["rgb(50,50,50)"] * surveys.shape[0]
values = []
for val in surveys["sex"]: 
#     print(val)
    if str(val) == "1.0" or str(val) == "M": # M
        values.append(list(count_sex.keys())[0])
    elif str(val) == "2.0" or str(val) == "F": # F
        values.append(list(count_sex.keys())[1])
    else: # else
        values.append(list(count_sex.keys())[2])
#     print(val, values[-1])
sexes = values[:]
print(len(sexes), Counter(sexes))

In [ ]:
surveys["sex"].value_counts()

In [ ]:
surveys[~surveys["sex"].isin(["M", "F"])]

In [ ]:
def star_chart_per_patient(cluster): 
    scatterpolars = []
#     count_sex = {"rgba(0,255,0,0.75)": 0, "rgba(0,0,255,0.75)": 0, "rgba(50,50,50,0.25)": 0}
#     count_sex = {"rgb(0,255,0)": 0, "rgb(0,0,255)": 0, "rgb(50,50,50)": 0}
    count_sex = {"rgb(0,0,255)": 0, "rgb(255,0,0)": 0, "rgb(50,50,50)": 0}
    for i in list(patient_by_symptom[patient_by_symptom["cluster"] == cluster].index)[:]: 
        scatterpolars.append(go.Scatterpolar(dictionary[i], # fill = "toself", fillcolor = sexes[i], 
                                             mode = "lines",            
                                             name = "Patient" + str(i+1), 
                                             line_color = sexes[i], 
#                                              line = {"opacity": 0.25}, 
#                                              line_color = "rgba(100,25,25,0.25)", 
                                            ))
        count_sex[sexes[i]] += 1
    fig = go.Figure(data = scatterpolars)
    fig.update_traces(opacity = 0.25)
    # fig.update_polars(radialaxis=dict(range=[0, 1]))
    count_sex_str = str(count_sex).replace(list(count_sex.keys())[0], "M (blue)").replace(list(count_sex.keys())[1], "F (red)").replace(list(count_sex.keys())[2], "nan (gray)")
#     count_sex_str = ""
    fig.update_layout(title = ".\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\tCluster" + str(cluster) + ": " + str(len(scatterpolars)) + " patients " + count_sex_str)
    return fig

star_chart_per_patient(1)

In [ ]:
star_chart_per_patient(2)

In [ ]:
star_chart_per_patient(3)

In [ ]:
star_chart_per_patient(4)

In [ ]:
star_chart_per_patient(5)

In [ ]:
star_chart_per_patient(6)

In [ ]:
star_chart_per_patient(7)

In [ ]:
star_chart_per_patient(8)

In [ ]:
star_chart_per_patient(9)

In [ ]:
star_chart_per_patient(10)

In [ ]:
star_chart_per_patient(11)

In [ ]:
star_chart_per_patient(12)

In [ ]:
star_chart_per_patient(13)

In [ ]:
star_chart_per_patient(14)

In [ ]:
star_chart_per_patient(15)

In [ ]:
star_chart_per_patient(16)

# Star Charts Median

In [ ]:
patient_by_symptom["sex"] = surveys["sex"]
patient_by_symptom

In [ ]:
def star_chart_median(cluster): 
    dictionary = {}
    symptoms_subset = list(cluster_topsymptoms["C" + str(cluster)])[:6]
    cluster_subset = patient_by_symptom[patient_by_symptom["cluster"] == cluster]
#     return cluster_subset
#     cluster_subset["sex"] = "M"
    for sex in ["F", "M"]: 
        cluster_sex_subset = cluster_subset[cluster_subset["sex"] == sex]
        cluster_sex_subset = cluster_sex_subset.median()[symptoms_subset]
        cluster_sex_subset = dict(cluster_sex_subset)
        key = f"{sex} (in cluster)"
        dictionary[key] = dict(r = list(cluster_sex_subset.values()), theta = list(cluster_sex_subset.keys()))
        dictionary[key]["r"].append(dictionary[key]["r"][0])
        dictionary[key]["theta"].append(dictionary[key]["theta"][0])
    cluster_subset = patient_by_symptom[patient_by_symptom["cluster"] != cluster]
#     cluster_subset["sex"] = "M"
    for sex in ["F", "M"]: 
        cluster_sex_subset = cluster_subset[cluster_subset["sex"] == sex]
        cluster_sex_subset = cluster_sex_subset.median()[symptoms_subset]
        cluster_sex_subset = dict(cluster_sex_subset)
        key = f"{sex} (global)"
        dictionary[key] = dict(r = list(cluster_sex_subset.values()), theta = list(cluster_sex_subset.keys()))
        dictionary[key]["r"].append(dictionary[key]["r"][0])
        dictionary[key]["theta"].append(dictionary[key]["theta"][0])

    scatterpolars = []
    scatterpolars.append(go.Scatterpolar(dictionary["F (in cluster)"], mode = "lines", 
                                         name = "F (in cluster)", line_color = "rgba(255,0,0,0.25)", ))
    scatterpolars.append(go.Scatterpolar(dictionary["M (in cluster)"], mode = "lines", 
                                         name = "M (in cluster)", line_color = "rgba(0,0,255,0.25)", ))
    scatterpolars.append(go.Scatterpolar(dictionary["F (global)"], mode = "lines", line = dict(dash = "dash"), 
                                         name = "F (global)", line_color = "rgba(255,0,0,0.25)", ))
    scatterpolars.append(go.Scatterpolar(dictionary["M (global)"], mode = "lines", line = dict(dash = "dash"), 
                                         name = "M (global)", line_color = "rgba(0,0,255,0.25)", ))
    fig = go.Figure(data = scatterpolars)
    count_sex_str = ""
    fig.update_layout(title = ".\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\tCluster" + str(cluster) + ": " + str(patient_by_symptom[patient_by_symptom["cluster"] == cluster].shape[0]) + " patients " + count_sex_str)
    return fig
star_chart_median(1)

In [ ]:
star_chart_median(2)

In [ ]:
star_chart_median(3)

In [ ]:
star_chart_median(4)

In [ ]:
star_chart_median(5)

In [ ]:
star_chart_median(6)

In [ ]:
star_chart_median(7)

In [ ]:
star_chart_median(8)

In [ ]:
star_chart_median(9)

In [ ]:
star_chart_median(10)

In [ ]:
star_chart_median(11)

In [ ]:
star_chart_median(12)

In [ ]:
star_chart_median(13)

In [ ]:
star_chart_median(14)

In [ ]:
star_chart_median(15)

In [ ]:
star_chart_median(16)

# Getting endotypes

In [ ]:
cluster_topsymptoms = cluster_topsymptoms.head(6)
cluster_topsymptoms_dict = {}
for col in cluster_topsymptoms.columns: 
    cluster_topsymptoms_dict[int(col.replace("C", ""))] = list(cluster_topsymptoms[col])
cluster_topsymptoms_dict

In [ ]:
color_mapping_rev = dict(zip(color_mapping.values(), color_mapping.keys()))
cluster_toporgans_dict = {}
for cluster in cluster_topsymptoms_dict.keys(): 
    toporgans = {}
    for sym in cluster_topsymptoms_dict[cluster]: 
        val = color_mapping_rev[symptoms_ordered_colors[symptoms_ordered.index(sym)]]
        if val not in toporgans: toporgans[val] = 1
        else: toporgans[val] += 1
        if len(toporgans) == 3: break
    cluster_toporgans_dict[cluster] = list(np.unique(list(toporgans.keys())))
cluster_toporgans_dict

In [ ]:
ax = sns.clustermap(cluster_by_symptom, vmin = 0, col_cluster = True, row_cluster = False, row_colors = symptoms_ordered_colors, 
                    standard_scale = 0, 
                    xticklabels = 1, yticklabels = 1, cbar_kws = {'label': ''}, 
                    cmap = "Greys", linecolor = "black", linewidth = 0.5, square = True)
plt.legend(handles, color_mapping, title='Symptom Groups', facecolor = "white", 
           bbox_to_anchor=(0.89, 1.01), bbox_transform=plt.gcf().transFigure)
a = plt.setp(ax.ax_heatmap.get_xticklabels(), rotation=90, size = 12)
a = plt.title("row_scale") # 0
print(f'{file.split("Subphenotyping-for-PASC_test/")[-1]}')

In [ ]:
# printing top organ systems in cluster order
for key in [9, 8, 11, 14, 16, 13, 1, 12, 15, 6, 7, 10, 3, 4, 2, 5]: 
    print(key, cluster_toporgans_dict[key])

In [ ]:
endo_dict = {1:0, 2:1, 3:1, 4:1, 5:1, 6:2, 7:2, 8:3, 9:3, 10:4, 11:3, 12:5, 13:6, 14:7, 15:2, 16:7} # cardiff

In [ ]:
severity

# Final cluster assignments

In [ ]:
patients = pd.read_csv("data_cleaned/cardiff_data_all_n318.csv")
patients["id"] = [val.replace("0.", "CO").replace("1.", "CA") for val in patients["id"]]
values = []
for val in patients["id"]: 
    if len(val) == 4: values.append(val + "0")
    elif len(val) == 3: values.append(val + "00")
    elif len(val) == 2: values.append(val + "000")
    else: values.append(val)
patients["id"] = values
patients = patients.drop(297)
metadata = pd.read_csv("data_cleaned/metadata_cardiff_n338.csv")
patients = patients.merge(metadata[["id", "sex"]], on = "id", how = "left")
patients["id"].value_counts()
patients["cluster"] = clusters
dictionary = dict(zip(severity["cluster"], severity["color"]))
patients["severity"] = [dictionary[val] for val in patients["cluster"]]
dictionary = {"green": "mild", "yellow": "moderate", "red": "severe"}
patients["severity"] = [dictionary[val] for val in patients["severity"]]
dictionary = dict(zip(severity["cluster"], severity["sum"]))
patients["severity_score"] = [dictionary[val] for val in patients["cluster"]]
patients["top6symptoms"] = [cluster_topsymptoms_dict[val] for val in patients["cluster"]]
patients["top3organs"] = [cluster_toporgans_dict[val] for val in patients["cluster"]]
patients["endotype"] = ["E" + str(endo_dict[val]) for val in patients["cluster"]]
columns = ["id", "group"]
columns.extend(["cluster", "severity"])
columns.extend(list(patients.columns)[2:-2])
columns = ['id', 'group', 'sex', 'cluster', 'endotype', 'severity', 'severity_score', 'top6symptoms', 'top3organs']
#, 'chest_pain', 'palpitations_heart_racing', 'poor_concentration', 'breathlessness_shortness_of_breath', 'cough', 'fevers_did_you_feel_hot', 'headache', 'abdominal_pain', 'change_in_smell', 'dizziness_or_vertigo', 'muscle_pain', 'poor_balance_unsteadiness', 'sleep_disturbance', 'increased_anxiety_worry', 'low_mood_lack_of_enjoyment_in_normal_activities', 'nausea', 'vomiting', 'muscle_cramps', 'constipation', 'diarrhoea', 'feeling_faint_or_light_headed', 'loss_of_appetite', 'back_pain', 'chills_do_you_feel_unusually_cold', 'numbness_odd_or_loss_of_sensation', 'increased_body_odour_sweating', 'general_body_pain_e_g_arms_legs_or_joints', 'pain_during_sex_having_intercourse', 'rash', 'runny_or_congested_nose', 'sore_throat', 'high_temperature_when_you_need_it_measured']
patients = patients[columns]
patients.columns = ["Subject_ID", "Group", "Sex", "Cluster_ID", "Endotype_ID (defined by organ systems involved)", 
                    "Severity_Group (predicted by ML)", "severity_score", "Top6_signature_symptoms", "Top3_organ_systems"]
# patients.to_csv("cluster_assignments_cardiff_n318_18topics_16clusters_v3.csv", index = False)
patients

In [ ]:
)

# Subclustering

In [ ]:
def sorting(clusters): 
    # Sorting by cluster sizes
    dictionary = sorted(dict(Counter(clusters)).items(), key=lambda x:x[1])
    dictionary.reverse()
    # [(1, 183), (3, 94), (5, 88), (9, 68), (2, 59), (8, 52), (7, 50), (4, 39), (6, 36)]
    mapping = dict(zip([val[0] for val in dictionary], np.unique(clusters))) # new:original
    # {1: 1, 3: 2, 5: 3, 9: 4, 2: 5, 8: 6, 7: 7, 4: 8, 6: 9}
    clusters = [mapping[val] for val in clusters]
    return clusters

In [ ]:
cluster_topsymptoms

In [ ]:
patient_by_symptom

In [ ]:
cluster_df = pd.DataFrame()
cluster = 1
print(cluster)
subset = patient_by_symptom[patient_by_symptom["cluster"] == cluster]
topsymptoms = list(cluster_topsymptoms["C" + str(cluster)])
subset = subset[topsymptoms]
method = "KMeans"
method = "Birch"
for i in range(2, 11): 
    method_ = method + "_" + str(i)
#     clustering = sklearn.cluster.KMeans(n_clusters = i, random_state = 916).fit(subset)
    clustering = sklearn.cluster.Birch(n_clusters = i).fit(subset)
    cluster_df[method_] = sorting(clustering.labels_ + 1)
    print(len(np.unique(cluster_df[method_])))
    print(dict(Counter(cluster_df[method_])))
cluster_df

In [ ]:
from sklearn.metrics import silhouette_score 
import matplotlib.pyplot as plt

# Getting the best score per clustering method
# https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html
columns = []
dictionary = {}
silhouette_scores = pd.DataFrame()
for col in cluster_df.columns: 
    silhouette_avg = silhouette_score(subset, cluster_df[col])
    key = col.split("_")[0]
    if key not in dictionary or dictionary[key][1] < silhouette_avg: 
        dictionary[key] = (col, silhouette_avg)
    score = silhouette_score(subset, cluster_df[col])
    silhouette_scores = pd.concat([silhouette_scores, pd.DataFrame({"cluster": [col], "score": [score]})])
silhouette_scores

In [ ]:
n_clusters = 2
print(max(silhouette_scores["score"]))
a = plt.scatter(range(2, 11), silhouette_scores["score"])
a = plt.title(method)
a = plt.plot([n_clusters, n_clusters], [0, 0.8], color = "red")

In [ ]:
subset_norm = subset/subset.sum(1)[:,np.newaxis]
subset_norm = preprocessing.normalize(subset)
UMAP = umap.UMAP(metric='euclidean', n_neighbors=10, random_state=916)
embedding = UMAP.fit_transform(subset_norm)

In [ ]:
n_clusters = 2
clusters = get_clusters(clustering_method, n_clusters, subset)
# clusters = get_clusters(clustering_method, n_clusters, embedding)
print(Counter(clusters))
plot_clusters(clusters, embedding, f"{clustering_method} {n_clusters} clusters", colors, None, None)

In [ ]:
n_clusters = 2
clusters = get_clusters(clustering_method, n_clusters, subset)
# clusters = get_clusters(clustering_method, n_clusters, embedding)
print(Counter(clusters))
title = "Case vs Control"
patients = pd.read_csv("cardiff_data_all_n318.csv")
patients = patients.drop(298)
patients = patients.iloc[subset.index,:]
patients = list(patients[patients["group"] == "Control"].index)
len(patients)
plot_clusters(clusters, embedding, f"{title}: {len(patients)} Case out of {len(embedding)} patients", colors, topic_proportions__.reset_index(drop = True), patients)

In [ ]:
print(clusters)

In [ ]:
topic_proportions__ = topic_proportions_.iloc[subset.index,:]
topic_proportions__

In [ ]:
subset

In [ ]:
# topic_proportions
subset_ = subset.copy()
subset_["cluster"] = clusters
print(subset_["cluster"].value_counts())
subset_.groupby("cluster").mean()

In [ ]:
n_clusters = 2
# clusters = get_clusters(clustering_method, n_clusters, subset)
clusters = get_clusters(clustering_method, n_clusters, embedding)
print(Counter(clusters))
plot_clusters(clusters, embedding, f"{clustering_method} {n_clusters} clusters", colors, None, None)

In [ ]:
# embedding
subset_ = subset.copy()
subset_["cluster"] = clusters
print(subset_["cluster"].value_counts())
subset_.groupby("cluster").mean()